# Bài học 11 - Giao thức Đại lý tới Đại lý (A2A)


## Cài đặt


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity

In [ ]:
import os
import asyncio
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Giao thức A2A là gì?

The **Tác nhân-đến-Tác nhân (A2A) protocol** là một tiêu chuẩn mở cho phép các tác nhân AI giao tiếp,
khám phá lẫn nhau và hợp tác — ngay cả khi chúng được xây dựng trên các khung khác nhau hoặc được lưu trữ
bởi các dịch vụ khác nhau.

Các khái niệm chính:

- **Khám phá** – Các tác nhân công bố một *Thẻ Tác nhân* mô tả khả năng của họ, giúp
  những tác nhân khác (hoặc bộ điều phối) dễ dàng tìm được chuyên gia phù hợp cho một nhiệm vụ.
- **Truyền thông điệp** – Các tác nhân trao đổi các thông điệp có cấu trúc thông qua một giao thức chung, vì vậy một
  yêu cầu từ một tác nhân có thể được một tác nhân khác hiểu và thực hiện bất kể
  cài đặt nội bộ của nó.
- **Vòng đời tác vụ** – A2A định nghĩa các trạng thái như *submitted*, *working*, *completed*, và
  *failed*, cung cấp cho bộ điều phối cái nhìn đầy đủ về cách một tác vụ được ủy quyền đang tiến triển.

Trong bài học này chúng tôi mô phỏng sự hợp tác theo kiểu A2A bằng cách kết nối ba tác nhân du lịch chuyên biệt
vào một luồng công việc nơi mỗi tác nhân đóng góp chuyên môn của mình và chuyển kết quả cho người tiếp theo.


## Tạo các đại lý du lịch chuyên biệt


In [ ]:
currency_agent = await provider.create_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = await provider.create_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = await provider.create_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## Hợp tác đa tác nhân thông qua quy trình công việc

Chúng tôi kết nối ba tác nhân này thành một quy trình tuần tự phản ánh việc truyền tin A2A:

1. **CurrencyExchangeAgent** nhận yêu cầu của người dùng và đưa ra hướng dẫn về tiền tệ.
2. **ActivityPlannerAgent** nhận bối cảnh được mở rộng và bổ sung các đề xuất hoạt động.
3. **TravelManagerAgent** tổng hợp cả hai đầu vào thành một bản tóm tắt du lịch cuối cùng.


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Hiểu A2A trong môi trường sản xuất

Trong môi trường sản xuất, giao thức A2A mở khóa các kịch bản mạnh mẽ xuyên dịch vụ:

| Khả năng | Mô tả |
|---|---|
| **Tương tác giữa các framework** | Một agent được xây dựng bằng một framework có thể ủy nhiệm tác vụ cho một agent được xây dựng bằng bất kỳ framework nào tuân thủ A2A, cho phép khả năng tương tác thực sự giữa các tổ chức. |
| **Ranh giới dịch vụ** | Các agent có thể chạy trong các microservice riêng biệt, vùng đám mây khác nhau, hoặc thậm chí các tổ chức khác nhau trong khi vẫn hợp tác liền mạch. |
| **Khám phá động** | Một bộ điều phối có thể truy vấn một registry Agent Card tại thời gian chạy để tìm chuyên gia phù hợp nhất cho một nhiệm vụ phụ cụ thể. |
| **Streaming & thông báo đẩy** | A2A hỗ trợ Server-Sent Events (SSE) để cập nhật tiến độ theo thời gian thực và thông báo đẩy cho các tác vụ chạy dài. |

Luồng công việc mà chúng ta đã xây dựng ở trên là một phiên bản đơn giản, chạy trong cùng một tiến trình của mẫu này. In a real
deployment each agent would expose an HTTP endpoint, publish an Agent Card, and communicate
via the A2A JSON-RPC protocol.


## Tóm tắt

Trong bài học này bạn đã học:

1. **Giao thức A2A là gì** — một tiêu chuẩn mở cho phát hiện giữa các tác nhân, nhắn tin, và quản lý tác vụ.
2. **Cách tạo các tác nhân chuyên biệt** — một tác nhân Chuyển đổi Tiền tệ, một tác nhân Lập kế hoạch Hoạt động, và một bộ điều phối Quản lý Du lịch.
3. **Cách nối các tác nhân vào một quy trình công việc** — sử dụng `WorkflowBuilder` để mô hình hóa việc chuyển tin theo thứ tự giữa các tác nhân.
4. **Cách A2A hoạt động trong môi trường sản xuất** — cho phép hợp tác giữa các framework và dịch vụ với khả năng khám phá động và cập nhật theo luồng.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Miễn trừ trách nhiệm:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI Co-op Translator (https://github.com/Azure/co-op-translator). Mặc dù chúng tôi nỗ lực đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc không chính xác. Tài liệu gốc bằng ngôn ngữ ban đầu nên được coi là nguồn tham chiếu chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp do con người thực hiện. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc diễn giải sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
